# ILN05 - Envío 2: Transformers con RoBERTa

En este notebook se entrena un modelo basado en `roberta-base` para la tarea *Movie Genre Identification Task*.

El problema se plantea como clasificación multi-etiqueta: una película puede pertenecer a uno o varios géneros. Para ello se usa `AutoModelForSequenceClassification` con `problem_type="multi_label_classification"`, lo que permite entrenar con etiquetas binarias por género. Después del entrenamiento se ajusta un umbral independiente por clase sobre el conjunto de validación.


## 1. Instalación e importación de librerías

La celda de instalación puede dejarse comentada si el entorno ya tiene las dependencias necesarias. En Colab o en un entorno nuevo conviene ejecutarla una vez.


In [ ]:
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    accuracy_score,
    hamming_loss,
    precision_recall_fscore_support,
    f1_score,
    classification_report,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)


## 2. Configuración general

Se fija la semilla para mejorar la reproducibilidad. El modelo elegido es `roberta-base`, con longitud máxima de entrada de 256 tokens.


In [ ]:
SEED = 42
MODEL_NAME = "roberta-base"
MAX_LEN = 256
NUM_EPOCHS = 4
LEARNING_RATE = 2e-5


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    if hasattr(torch.backends, "mps"):
        torch.backends.mps.is_available()


set_seed(SEED)

print("CUDA disponible:", torch.cuda.is_available())
if hasattr(torch.backends, "mps"):
    print("MPS disponible:", torch.backends.mps.is_available())


## 3. Carga de datos

Se descomprime el dataset si los CSV no existen todavía. El fichero de entrenamiento incluye la columna `genre`, mientras que el fichero de test solo contiene título y descripción.


In [ ]:
train_path = Path("dataset_train.csv")
test_path = Path("dataset_test.csv")
zip_path = Path("Dataset-Movies.zip")

if not train_path.exists() or not test_path.exists():
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(".")
    print("Dataset descomprimido.")
else:
    print("Los CSV ya existen. No se descomprime de nuevo.")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Train:", train_df.shape)
print("Test:", test_df.shape)

display(train_df.head())
display(test_df.head())


## 4. Preparación de etiquetas y texto

Los géneros se separan por coma y se transforman en una matriz binaria con `MultiLabelBinarizer`. Como entrada del transformer se combina el título y la descripción en un único texto.


In [ ]:
train_df["genre_list"] = train_df["genre"].apply(
    lambda x: [genre.strip() for genre in x.split(",")]
)

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(train_df["genre_list"])
genres = list(mlb.classes_)

print("Géneros:")
print(genres)
print("\nShape de Y:", Y.shape)

train_df["text"] = (
    "Title: " + train_df["movie_name"].fillna("") +
    ". Plot: " + train_df["description"].fillna("")
)

test_df["text"] = (
    "Title: " + test_df["movie_name"].fillna("") +
    ". Plot: " + test_df["description"].fillna("")
)

display(train_df[["movie_name", "genre", "text"]].head())


## 5. División entrenamiento-validación

Se reserva un 20% del entrenamiento como validación interna. Esta partición se usa para evaluar el modelo y para ajustar los umbrales por clase.


In [ ]:
X = train_df["text"].tolist()
X_test = test_df["text"].tolist()

X_train, X_dev, y_train, y_dev = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=SEED,
)

print("X_train:", len(X_train))
print("X_dev:", len(X_dev))
print("y_train:", y_train.shape)
print("y_dev:", y_dev.shape)


## 6. Métricas auxiliares

Durante el entrenamiento se evalúa con umbral fijo de 0.5. Después se ajustan umbrales independientes por clase para mejorar el F1 macro.


In [ ]:
def compute_metrics_official(y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )
    acc = accuracy_score(y_true, y_pred)
    hl = hamming_loss(y_true, y_pred)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "hamming_loss": hl,
    }


def compute_metrics_debug(y_true, y_pred):
    metrics = compute_metrics_official(y_true, y_pred)
    metrics["avg_labels_pred"] = y_pred.sum(axis=1).mean()
    metrics["avg_labels_true"] = y_true.sum(axis=1).mean()
    return metrics


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def compute_metrics_trainer(eval_pred):
    logits, labels = eval_pred
    probs = sigmoid(logits)
    preds = (probs >= 0.5).astype(int)
    return compute_metrics_official(labels, preds)


## 7. Dataset para PyTorch

Esta clase tokeniza cada texto con el tokenizer de RoBERTa. En entrenamiento y validación incluye las etiquetas; en test permite trabajar sin etiquetas reales.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class MovieGenreDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        item = {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
        }

        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)

        return item


train_dataset = MovieGenreDataset(X_train, y_train, tokenizer, MAX_LEN)
dev_dataset = MovieGenreDataset(X_dev, y_dev, tokenizer, MAX_LEN)
test_dataset = MovieGenreDataset(X_test, labels=None, tokenizer=tokenizer, max_len=MAX_LEN)

print(len(train_dataset), len(dev_dataset), len(test_dataset))


## 8. Modelo RoBERTa multi-etiqueta

Se carga `roberta-base` con una cabeza de clasificación de 18 salidas, una por género. Los avisos de pesos inesperados o recién inicializados son normales porque se está adaptando un modelo preentrenado a una nueva tarea de clasificación.


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(genres),
    problem_type="multi_label_classification",
)


## 9. Configuración de entrenamiento

Se entrena durante 4 épocas, evaluando y guardando un checkpoint al final de cada época. Se carga automáticamente el mejor modelo según F1 de validación. `dataloader_pin_memory=False` evita avisos innecesarios cuando se usa MPS en Mac.


In [ ]:
training_args = TrainingArguments(
    output_dir="./roberta_base_results",
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1,
    report_to="none",
    dataloader_pin_memory=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics_trainer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)


## 10. Entrenamiento y evaluación con umbral 0.5

La primera evaluación se realiza con umbral fijo de 0.5 para todas las clases. Después se ajustarán umbrales específicos por género.


In [ ]:
trainer.train()


In [ ]:
eval_results = trainer.evaluate()
display(eval_results)
print("Dispositivo usado por Trainer:", trainer.args.device)


## 11. Ajuste de umbrales por clase

En clasificación multi-etiqueta, usar 0.5 para todas las clases puede no ser óptimo. Por eso se calcula el umbral que maximiza el F1 de cada género en validación.


In [ ]:
dev_outputs = trainer.predict(dev_dataset)
dev_logits = dev_outputs.predictions
dev_probs = sigmoid(dev_logits)

print("Shape probabilidades dev:", dev_probs.shape)


def find_best_thresholds(y_true, y_scores, thresholds=np.arange(0.05, 0.96, 0.01)):
    best_thresholds = []
    best_f1s = []

    for j in range(y_true.shape[1]):
        best_t = 0.5
        best_f1 = 0.0

        for t in thresholds:
            y_pred_j = (y_scores[:, j] >= t).astype(int)
            f1 = f1_score(y_true[:, j], y_pred_j, zero_division=0)

            if f1 > best_f1:
                best_f1 = f1
                best_t = t

        best_thresholds.append(best_t)
        best_f1s.append(best_f1)

    return np.array(best_thresholds), np.array(best_f1s)


best_thresholds_roberta, best_f1s_roberta = find_best_thresholds(y_dev, dev_probs)

thresholds_roberta_df = pd.DataFrame({
    "genre": genres,
    "threshold": best_thresholds_roberta,
    "dev_f1_class": best_f1s_roberta,
    "support_dev": y_dev.sum(axis=0),
}).sort_values("genre")

display(thresholds_roberta_df)


## 12. Evaluación final en validación con umbrales ajustados

Se recalculan las métricas aplicando los umbrales por clase. Esta es la evaluación principal del modelo RoBERTa.


In [ ]:
y_pred_roberta_thr = (dev_probs >= best_thresholds_roberta).astype(int)

metrics_roberta_thr = compute_metrics_debug(y_dev, y_pred_roberta_thr)
display(metrics_roberta_thr)

print(classification_report(
    y_dev,
    y_pred_roberta_thr,
    target_names=genres,
    zero_division=0,
))


## 13. Predicción sobre test y generación del envío

Se predice el conjunto de test con el mejor checkpoint cargado por `Trainer`. Después se aplican los umbrales ajustados en validación y se genera el CSV final. Si alguna película queda sin género asignado, se selecciona el género con mayor probabilidad.


In [ ]:
test_outputs = trainer.predict(test_dataset)
test_logits = test_outputs.predictions
test_probs_roberta = sigmoid(test_logits)

print("Shape probabilidades test:", test_probs_roberta.shape)

test_pred_bin_roberta = (test_probs_roberta >= best_thresholds_roberta).astype(int)

for i in range(test_pred_bin_roberta.shape[0]):
    if test_pred_bin_roberta[i].sum() == 0:
        best_class = test_probs_roberta[i].argmax()
        test_pred_bin_roberta[i, best_class] = 1

test_pred_labels_roberta = mlb.inverse_transform(test_pred_bin_roberta)
test_pred_genres_roberta = [", ".join(labels) for labels in test_pred_labels_roberta]

submission_roberta_df = pd.DataFrame({
    "movie_name": test_df["movie_name"],
    "description": test_df["description"],
    "genre": test_pred_genres_roberta,
})

display(submission_roberta_df.head())
print(submission_roberta_df.shape)


## 14. Guardado y comprobación del ZIP

El fichero `dataset_test_preds.csv` se comprime en un ZIP con el formato esperado para el envío.


In [ ]:
submission_path = "dataset_test_preds.csv"
zip_output_path = "ILN05-RoBERTaBase-PerClassTH.zip"

submission_roberta_df.to_csv(submission_path, index=False)

with zipfile.ZipFile(zip_output_path, "w") as zipf:
    zipf.write(submission_path)

check_df = pd.read_csv(submission_path)

print(check_df.shape)
print(check_df.columns.tolist())
print("Nulos en genre:", check_df["genre"].isna().sum())
print("Géneros vacíos:", (check_df["genre"].str.len() == 0).sum())

with zipfile.ZipFile(zip_output_path, "r") as zipf:
    print(zipf.namelist())

display(check_df.head())
